# 03 Build Series and Adjust Frequencies

Resample everything to monthly averages, merge, and compute the spread, the Cleveland Fed decomposition, rate volatility and the change since January 2021. Writes `data/processed/` and `results/tables/`.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent  

In [11]:
raw_dir = ROOT / "data" / "raw"
series_ids = ["MORTGAGE30US", "DGS10", "DFII10", "T10YIE", "WSHOMCB",
              "EXPINF10YR", "REAINTRATREARAT10Y", "TENEXPCHAREARISPRE", "TENEXPCHAINFRISPRE"]

# Monthly average of every series
monthly = []
for sid in series_ids:
    s = pd.read_csv(raw_dir / f"{sid}.csv", index_col="date", parse_dates=True)
    monthly.append(s.resample("MS").mean())

df = pd.concat(monthly, axis=1, join="inner")

# Rename for readability
NAMES = {
    "MORTGAGE30US": "FRM_30Y",
    "DGS10": "Treasury_10Y",
    "DFII10": "TIPS_10Y",
    "T10YIE": "Breakeven_10Y",
    "WSHOMCB": "Fed_MBS_holdings",
    "EXPINF10YR": "exp_infl_10Y",
    "REAINTRATREARAT10Y": "real_rate_10Y",
    "TENEXPCHAREARISPRE": "real_risk_prem_10Y",
    "TENEXPCHAINFRISPRE": "infl_risk_prem_10Y"
}
df = df.rename(columns = NAMES)

# Drop the current month b/c its average would cover only a few days
this_month = pd.Timestamp.today().to_period("M").to_timestamp()
df = df[df.index < this_month]
df

,FRM_30Y,Treasury_10Y,TIPS_10Y,Breakeven_10Y,Fed_MBS_holdings,exp_infl_10Y,real_rate_10Y,real_risk_prem_10Y,infl_risk_prem_10Y
date,,,,,,,,,
2003-01-01,5.9160,4.048571,2.294286,1.754286,0.00,2.243939,1.488202,1.196914,0.386633
2003-02-01,5.8425,3.902632,1.990000,1.912632,0.00,2.245564,1.466379,1.177252,0.373463
2003-03-01,5.7450,3.807143,1.944286,1.862857,0.00,2.078386,1.223591,1.151881,0.369566
2003-04-01,5.8125,3.958571,2.183810,1.774762,0.00,2.116690,1.309618,1.192092,0.385485
2003-05-01,5.4840,3.569048,1.908095,1.660952,0.00,2.117078,1.370721,1.214038,0.395925
...,...,...,...,...,...,...,...,...,...
2026-05-01,6.4425,4.484000,2.039000,2.445000,1976141.75,2.487485,1.628008,1.216583,0.338697
2026-06-01,6.4900,4.470476,2.179048,2.291429,1963990.00,2.465424,1.915091,1.301859,0.371442
2026-07-01,6.5420,4.599545,2.353182,2.246364,1944155.20,2.435820,2.073276,1.309481,0.413586


In [12]:
# Mortgage-Treasury spread, percentage points
df["spread"] = df["FRM_30Y"] - df["Treasury_10Y"]

# Cleveland Fed: expected path of real short rates = real rate - real risk premium
df["exp_real_path_10Y"] = df["real_rate_10Y"] - df["real_risk_prem_10Y"]

# Model residual: residual = 10-Year Treasury yield - Cleveland Fed's estimated yield
df["cleveland_resid"] = df["Treasury_10Y"] - (df["real_rate_10Y"] + df["exp_infl_10Y"] + df["infl_risk_prem_10Y"])

# Fed MBS holdings: millions -> trillions of dollars, and 12-month change
df["mbs_trn"] = df["Fed_MBS_holdings"] / 1e6
df["mbs_chg12_trn"] = df["mbs_trn"].diff(12)

# Realized 10-year yield volatility: 
# 1. daily change in bp
# 2. rolling 63-trading-day (~3 months) std, annualized with sqrt(252)
dgs10_daily = pd.read_csv(raw_dir / "DGS10.csv", index_col="date", parse_dates=True)["DGS10"].dropna()
vol_daily = (dgs10_daily.diff() * 100).rolling(63).std() * np.sqrt(252)
df["vol10_bp"] = vol_daily.resample("MS").mean()

out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "monthly.csv")

print(f"{df.index.min():%Y-%m} to {df.index.max():%Y-%m}, {len(df)} months")

2003-01 to 2026-09, 285 months


In [13]:
# Define the trough as Jan 1, 2021
BASE = pd.Timestamp("2021-01-01")

# 30Y Mortgage rate = 10Y Treasury Yield + Spread
# 1. 10Y Treasury Yield = (exp_real_path + 10Y_infl_expec) + (real RP + inflation RP) + residual
# 1. + spread
pieces = ["exp_real_path_10Y", "exp_infl_10Y", "real_risk_prem_10Y", "infl_risk_prem_10Y", "cleveland_resid", "spread"]

contrib = df.loc[BASE:, pieces] - df.loc[BASE, pieces]
contrib["total"] = df.loc[BASE:, "FRM_30Y"] - df.loc[BASE, "FRM_30Y"]

gap = (contrib[pieces].sum(axis=1) - contrib["total"]).abs().max()
assert gap < 1e-9, gap
contrib.to_csv(out_dir / "contributions.csv")

In [14]:
tab_dir = ROOT / "results" / "tables"
tab_dir.mkdir(parents=True, exist_ok=True)

peak = df.loc[BASE:, "FRM_30Y"].idxmax()
latest = df.index.max()

# headline.csv: levels at the base, the post-2021 peak and the latest month, and the changes
cols = ["FRM_30Y", "Treasury_10Y"] + pieces + ["TIPS_10Y", "Breakeven_10Y"]
headline = pd.DataFrame({
    f"level_{BASE:%Y-%m}": df.loc[BASE, cols],
    f"level_{peak:%Y-%m}": df.loc[peak, cols],
    f"level_{latest:%Y-%m}": df.loc[latest, cols],
})
headline["chg_to_peak"] = headline.iloc[:, 1] - headline.iloc[:, 0]
headline["chg_to_latest"] = headline.iloc[:, 2] - headline.iloc[:, 0]
headline.round(2).to_csv(tab_dir / "headline.csv")

# spread_stats.csv
s = df["spread"]
spread_stats = pd.Series({
    "avg_2010_2019": round(s["2010":"2019"].mean(), 2),
    "peak_since_2021": round(s[BASE:].max(), 2),
    "peak_since_2021_date": f"{s[BASE:].idxmax():%Y-%m}",
    f"latest_{latest:%Y-%m}": round(s.iloc[-1], 2),
}, name="spread_pp")
spread_stats.to_csv(tab_dir / "spread_stats.csv")

# residual.csv: size of the Cleveland model residual
r = df["cleveland_resid"]
residual = pd.Series({
    "mean_full_sample": r.mean(),
    "max_abs_full_sample": r.abs().max(),
    "mean_since_2021": r[BASE:].mean(),
    "max_abs_since_2021": r[BASE:].abs().max(),
    "chg_base_to_latest": r.iloc[-1] - r[BASE],
}, name="pp").round(2)
residual.to_csv(tab_dir / "residual.csv")

print(headline.round(2).to_string(), "\n")
print(spread_stats.to_string(), "\n")
print(residual.to_string())

                    level_2021-01  level_2023-10  level_2026-09  chg_to_peak  chg_to_latest
FRM_30Y                      2.74           7.62           6.86         4.88           4.13
Treasury_10Y                 1.08           4.80           4.99         3.72           3.91
exp_real_path_10Y           -1.40           0.62           0.91         2.03           2.31
exp_infl_10Y                 1.38           2.36           2.57         0.98           1.19
real_risk_prem_10Y           1.17           1.46           1.34         0.29           0.17
infl_risk_prem_10Y           0.39           0.46           0.46         0.07           0.07
cleveland_resid             -0.45          -0.10          -0.29         0.35           0.17
spread                       1.65           2.82           1.87         1.17           0.22
TIPS_10Y                    -1.00           2.41           2.64         3.41           3.64
Breakeven_10Y                2.08           2.39           2.35         0.30    

In [15]:
# contrib_shares.csv: three windows, each answering one question
#   peak:   Jan 2021 -> post-2021 peak    "what built the high?"
#   latest: Jan 2021 -> latest month      "what's left?"
#   ytd:    Jan -> latest month, this year "what's moving now?"
# Each piece's change in pp and as % of the mortgage rate's change over the same window.
# Shares can be negative or exceed 100% when pieces offset each other.
ytd_start = pd.Timestamp(f"{latest.year}-01-01")
windows = {
    f"peak_{BASE:%Y-%m}_to_{peak:%Y-%m}": (BASE, peak),
    f"latest_{BASE:%Y-%m}_to_{latest:%Y-%m}": (BASE, latest),
    f"ytd_{ytd_start:%Y-%m}_to_{latest:%Y-%m}": (ytd_start, latest),
}

labels = {
    "exp_real_path_10Y": "Expected real short rates",
    "exp_infl_10Y": "Expected inflation",
    "real_risk_prem_10Y": "Real risk premium",
    "infl_risk_prem_10Y": "Inflation risk premium",
    "cleveland_resid": "Model residual",
    "spread": "Mortgage-Treasury spread",
    "FRM_30Y": "Total: 30-year mortgage rate",
}
rows = pieces + ["FRM_30Y"]

shares = pd.DataFrame({"label": [labels[r] for r in rows]}, index=rows)
for name, (start, end) in windows.items():
    chg = df.loc[end, rows] - df.loc[start, rows]
    assert abs(chg[pieces].sum() - chg["FRM_30Y"]) < 1e-9     
    shares[f"{name}_pp"] = chg.round(2)
    shares[f"{name}_pct"] = (100 * chg / chg["FRM_30Y"]).round(0)

shares.to_csv(tab_dir / "contrib_shares.csv")
shares

,label,peak_2021-01_to_2023-10_pp,peak_2021-01_to_2023-10_pct,latest_2021-01_to_2026-09_pp,latest_2021-01_to_2026-09_pct,ytd_2026-01_to_2026-09_pp,ytd_2026-01_to_2026-09_pct
exp_real_path_10Y,Expected real short rates,2.03,41.0,2.31,56.0,0.49,65.0
exp_infl_10Y,Expected inflation,0.98,20.0,1.19,29.0,0.24,32.0
real_risk_prem_10Y,Real risk premium,0.29,6.0,0.17,4.0,0.09,11.0
infl_risk_prem_10Y,Inflation risk premium,0.07,1.0,0.07,2.0,0.01,1.0
cleveland_resid,Model residual,0.35,7.0,0.17,4.0,-0.05,-7.0
spread,Mortgage-Treasury spread,1.17,24.0,0.22,5.0,-0.02,-2.0
FRM_30Y,Total: 30-year mortgage rate,4.88,100.0,4.13,100.0,0.76,100.0


In [16]:
# model_crosscheck.csv: does the split depend on the model?
# Same three windows as contrib_shares.csv, change in pp. ACM is the NY Fed Adrian-Crump-Moench model:
#   ACMRNY10 = expected (risk-neutral) nominal short-rate path, ACMTP10 = term premium.
# Cleveland's nominal expected path = expected real path + expected inflation; its premium = real RP + inflation RP.
acm = pd.read_csv(raw_dir / "acm.csv", index_col="date", parse_dates=True).resample("MS").mean()
acm = acm[acm.index < this_month]

rows = {
    "Treasury 10y (DGS10)":              df["Treasury_10Y"],
    "Cleveland: expected nominal path":  df["exp_real_path_10Y"] + df["exp_infl_10Y"],
    "Cleveland: premia (real + infl)":   df["real_risk_prem_10Y"] + df["infl_risk_prem_10Y"],
    "Cleveland: model residual":         df["cleveland_resid"],
    "Cleveland: expected real path":     df["exp_real_path_10Y"],
    "Cleveland: expected inflation":     df["exp_infl_10Y"],
    "ACM: expected nominal path":        acm["ACMRNY10"],
    "ACM: term premium":                 acm["ACMTP10"],
    "ACM: fitted 10y yield":             acm["ACMY10"],
    "TIPS: real yield (DFII10)":         df["TIPS_10Y"],
    "TIPS: breakeven (T10YIE)":          df["Breakeven_10Y"],
}
x = pd.DataFrame(rows)
windows = {f"peak_{BASE:%b%y}_to_{peak:%b%y}": (BASE, peak),
           f"latest_{BASE:%b%y}_to_{latest:%Y-%m}": (BASE, latest),
           f"ytd_{latest:%Y}": (pd.Timestamp(f"{latest.year}-01-01"), latest)}
crosscheck = pd.DataFrame({name: x.loc[end] - x.loc[start] for name, (start, end) in windows.items()}).round(2)
crosscheck.to_csv(tab_dir / "model_crosscheck.csv")
print(crosscheck.to_string())


                                  peak_Jan21_to_Oct23  latest_Jan21_to_2026-09  ytd_2026
Treasury 10y (DGS10)                             3.72                     3.91      0.78
Cleveland: expected nominal path                 3.01                     3.51      0.73
Cleveland: premia (real + infl)                  0.36                     0.23      0.09
Cleveland: model residual                        0.35                     0.17     -0.05
Cleveland: expected real path                    2.03                     2.31      0.49
Cleveland: expected inflation                    0.98                     1.19      0.24
ACM: expected nominal path                       2.87                     2.57      0.72
ACM: term premium                                0.91                     1.31     -0.02
ACM: fitted 10y yield                            3.78                     3.89      0.70
TIPS: real yield (DFII10)                        3.41                     3.64      0.73
TIPS: breakeven (T10Y